In [2]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from data_profiling import ProfileReport
import numpy as np

c:\Users\ahaly\OneDrive\Desktop\Programming for ADS\Project 2\project2\project2_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**5.2. Prepare the data and mine the rules**

*Load the data*

In [5]:
alzheimer_raw = pd.read_csv('../datasets/Alzheimer\'s_Disease_and_Healthy_Aging_Data_20261116.csv')

In [6]:
alzheimer_raw.head()

,RowId,YearStart,YearEnd,LocationAbbr,LocationDesc,Datasource,Class,Topic,Question,Data_Value_Unit,...,Stratification2,Geolocation,ClassID,TopicID,QuestionID,LocationID,StratificationCategoryID1,StratificationID1,StratificationCategoryID2,StratificationID2
0,BRFSS~2015~2015~66~Q35~TOC03~AGE~SEX,2015,2015,GU,Guam,BRFSS,Overall Health,Recent activity limitations in past month,Mean number of days with activity limitations ...,Number,...,Female,POINT (144.793731 13.444304),C01,TOC03,Q35,66,AGE,AGE_OVERALL,SEX,FEMALE
1,BRFSS~2015~2015~25~Q27~TMC03~AGE~SEX,2015,2015,MA,Massachusetts,BRFSS,Mental Health,Lifetime diagnosis of depression,Percentage of older adults with a lifetime dia...,%,...,Male,POINT (-72.08269067499964 42.27687047000046),C05,TMC03,Q27,25,AGE,65PLUS,SEX,MALE
2,BRFSS~2015~2015~9002~Q43~TOC11~AGE~SEX,2015,2015,MDW,Midwest,BRFSS,Overall Health,Arthritis among older adults,Percentage of older adults ever told they have...,%,...,Male,NaN,C01,TOC11,Q43,9002,AGE,5064,SEX,MALE
3,BRFSS~2015~2015~27~Q03~TMC01~AGE~SEX,2015,2015,MN,Minnesota,BRFSS,Mental Health,Frequent mental distress,Percentage of older adults who are experiencin...,%,...,Male,POINT (-94.79420050299967 46.35564873600049),C05,TMC01,Q03,27,AGE,AGE_OVERALL,SEX,MALE
4,BRFSS~2015~2015~29~Q43~TOC11~AGE~OVERALL,2015,2015,MO,Missouri,BRFSS,Overall Health,Arthritis among older adults,Percentage of older adults ever told they have...,%,...,NaN,POINT (-92.56630005299968 38.635790776000476),C01,TOC11,Q43,29,AGE,AGE_OVERALL,OVERALL,OVERALL


In [7]:
alzheimer = alzheimer_raw.copy()

*Examine the Data*

In [8]:
missing_info = pd.DataFrame({
    "Missing Values": alzheimer.isnull().sum(),
    "Missing Percentage": (alzheimer.isnull().mean() * 100).round(2)
})
missing_info

,Missing Values,Missing Percentage
RowId,0,0.00
YearStart,0,0.00
YearEnd,0,0.00
LocationAbbr,0,0.00
LocationDesc,0,0.00
Datasource,0,0.00
Class,0,0.00
Topic,0,0.00
Question,0,0.00
Data_Value_Unit,0,0.00


In [11]:
# Check complete duplicate rows
print("Duplicate rows (full record):", alzheimer.duplicated().sum())

# Check whether RowId is unique
print("Unique RowId values:", alzheimer["RowId"].nunique())
print("Duplicate RowId rows:", alzheimer.duplicated("RowId").sum())

# Inspect duplicate RowId records
print(
    alzheimer[alzheimer.duplicated("RowId", keep=False)]
    [["RowId", "YearStart", "LocationDesc",
      "QuestionID", "Stratification1", "Stratification2"]]
    .head(10)
)

Duplicate rows (full record): 0
Unique RowId values: 36046
Duplicate RowId rows: 248096
                                        RowId  YearStart   LocationDesc  \
0        BRFSS~2015~2015~66~Q35~TOC03~AGE~SEX       2015           Guam   
1        BRFSS~2015~2015~25~Q27~TMC03~AGE~SEX       2015  Massachusetts   
2      BRFSS~2015~2015~9002~Q43~TOC11~AGE~SEX       2015        Midwest   
3        BRFSS~2015~2015~27~Q03~TMC01~AGE~SEX       2015      Minnesota   
4    BRFSS~2015~2015~29~Q43~TOC11~AGE~OVERALL       2015       Missouri   
5      BRFSS~2015~2015~9001~Q18~TSC08~AGE~SEX       2015      Northeast   
6        BRFSS~2015~2015~40~Q03~TMC01~AGE~SEX       2015       Oklahoma   
7    BRFSS~2015~2015~72~Q03~TMC01~AGE~OVERALL       2015    Puerto Rico   
8  BRFSS~2015~2015~9003~Q43~TOC11~AGE~OVERALL       2015          South   
9        BRFSS~2015~2015~47~Q43~TOC11~AGE~SEX       2015      Tennessee   

  QuestionID    Stratification1 Stratification2  
0        Q35            Overall     

In [17]:
#Check the composite key

key_columns = [
    "YearStart",
    "YearEnd",
    "LocationID",
    "QuestionID",
    "StratificationID1",
    "StratificationID2"
]

print("Duplicate composite keys:",
      alzheimer.duplicated(subset=key_columns).sum())

Duplicate composite keys: 0


In [19]:
#Find all columns containing categorical/text data:

# Identify categorical columns
categorical_columns = alzheimer.select_dtypes(
    include=["object"]
).columns

print("Number of categorical columns:", len(categorical_columns))
print(categorical_columns.tolist())

Number of categorical columns: 24
['RowId', 'LocationAbbr', 'LocationDesc', 'Datasource', 'Class', 'Topic', 'Question', 'Data_Value_Unit', 'DataValueTypeID', 'Data_Value_Type', 'Data_Value_Footnote_Symbol', 'Data_Value_Footnote', 'StratificationCategory1', 'Stratification1', 'StratificationCategory2', 'Stratification2', 'Geolocation', 'ClassID', 'TopicID', 'QuestionID', 'StratificationCategoryID1', 'StratificationID1', 'StratificationCategoryID2', 'StratificationID2']


In [21]:
# Display value counts for all categorical columns
for col in categorical_columns:
    print("\n", "=" * 60)
    print("Column:", col)
    print(alzheimer[col].value_counts(dropna=False))


Column: RowId
RowId
BRFSS~2016~2016~9002~Q34~TOC09~AGE~RACE     15
BRFSS~2020~2020~19~Q19~TSC04~AGE~RACE       15
BRFSS~2017~2017~41~Q02~TNC02~AGE~RACE       15
BRFSS~2017~2017~44~Q16~TNC03~AGE~RACE       15
BRFSS~2017~2017~39~Q21~TAC03~AGE~RACE       15
                                            ..
BRFSS~2022~2022~50~Q12~TSC01~AGE~OVERALL     3
BRFSS~2022~2022~78~Q45~TOC13~AGE~OVERALL     3
BRFSS~2022~2022~55~Q07~TOC05~AGE~OVERALL     3
BRFSS~2022~2022~55~Q37~TGC02~AGE~OVERALL     3
BRFSS~2016~2016~02~Q03~TMC01~AGE~OVERALL     3
Name: count, Length: 36046, dtype: int64

Column: LocationAbbr
LocationAbbr
US      6132
WEST    6126
NRE     6102
MDW     6099
SOU     6030
OR      5916
NY      5787
UT      5368
OH      5311
HI      5164
MD      5132
GA      5124
MI      5113
NV      4950
ME      4949
TX      4947
MS      4939
VA      4936
WI      4928
WV      4856
TN      4844
PA      4810
SC      4807
AL      4807
NM      4805
KY      4795
OK      4779
AK      4763
IN      4761
LA      4

In [22]:
# Identify numerical columns

numerical_columns = alzheimer.select_dtypes(
    include=["int64", "float64"]
).columns

print("Numerical columns:")
print(numerical_columns.tolist())

# Descriptive statistics
print(alzheimer[numerical_columns].describe())

Numerical columns:
['YearStart', 'YearEnd', 'Data_Value', 'Data_Value_Alt', 'Low_Confidence_Limit', 'High_Confidence_Limit', 'LocationID']
           YearStart        YearEnd     Data_Value  Data_Value_Alt  \
count  284142.000000  284142.000000  192808.000000   192808.000000   
mean     2018.596065    2018.657735      37.676757       37.676757   
std         2.302815       2.360105      25.213484       25.213484   
min      2015.000000    2015.000000       0.000000        0.000000   
25%      2017.000000    2017.000000      15.900000       15.900000   
50%      2019.000000    2019.000000      32.800000       32.800000   
75%      2021.000000    2021.000000      56.900000       56.900000   
max      2022.000000    2022.000000     100.000000      100.000000   

       Low_Confidence_Limit  High_Confidence_Limit     LocationID  
count         192597.000000          192597.000000  284142.000000  
mean              33.027824              42.595333     800.322677  
std               24.29001

In [24]:
# Check the year range

print("YearStart values:")
print(sorted(alzheimer["YearStart"].unique()))

print("\nYearEnd values:")
print(sorted(alzheimer["YearEnd"].unique()))

print("\nRecords where YearStart differs from YearEnd:")
print((alzheimer["YearStart"] != alzheimer["YearEnd"]).sum())

YearStart values:
[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

YearEnd values:
[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

Records where YearStart differs from YearEnd:
9261


*EDA Report*

In [26]:

profile = ProfileReport(alzheimer,title="Alzheimer's Disease and Healthy Aging EDA Report")

# Save the report
profile.to_file("../outputs/Thirumurugan_Bocardo_part2_alzheimer_EDA.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\ahaly\OneDrive\Desktop\Programming for ADS\Project 2\project2\project2_env\Lib\site-packages\data_profiling\model\describe.py:85: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_start = datetime.utcnow()
Summarize dataset: 100%|██████████| 90/90 [00:36<00:00,  2.18it/s, Completed]                                           c:\Users\ahaly\OneDrive\Desktop\Programming for ADS\Project 2\project2\project2_env\Lib\site-packages\data_profiling\model\describe.py:190: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_end = datetime.utcnow()
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 53.16it/s]


Q1. List at least four data-quality issues you found in a) and b). For each one, give the column(s), the number of
rows affected, and how you will fix it in Section 4.3. This list is your cleaning plan.


Q2. Which of your issues did the EDA report flag (as alerts or in its statistics), and which did you find only with
pandas? Why can a profile report miss some problems?

*Clean and transform*

In [39]:
# Create a unique RecordID

key_columns = [
    "YearStart",
    "YearEnd",
    "LocationID",
    "QuestionID",
    "StratificationID1",
    "StratificationID2"
]

alzheimer["RecordID"] = (
    alzheimer[key_columns]
    .astype(str)
    .agg("_".join, axis=1)
)

print("Dataset shape:", alzheimer.shape)
print("Unique RecordIDs:", alzheimer["RecordID"].nunique())
print("Duplicate RecordIDs:", alzheimer["RecordID"].duplicated().sum())

Dataset shape: (284142, 32)
Unique RecordIDs: 284142
Duplicate RecordIDs: 0


In [40]:
# Create the cleaned dataset

# List of 50 states and Washington, DC
states_dc = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL",
    "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA",
    "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE",
    "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK",
    "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT",
    "VA", "WA", "WV", "WI", "WY", "DC"
]

# Filter the target population
clean = alzheimer[
    (alzheimer["Stratification1"] == "65 years or older") &
    (alzheimer["Stratification2"] == "Female") &
    (alzheimer["LocationAbbr"].isin(states_dc))
].copy()

print("Filtered dataset shape:", clean.shape)
display(clean.head())

Filtered dataset shape: (9885, 32)


,RowId,YearStart,YearEnd,LocationAbbr,LocationDesc,Datasource,Class,Topic,Question,Data_Value_Unit,...,Geolocation,ClassID,TopicID,QuestionID,LocationID,StratificationCategoryID1,StratificationID1,StratificationCategoryID2,StratificationID2,RecordID
11,BRFSS~2015~2015~49~Q27~TMC03~AGE~SEX,2015,2015,UT,Utah,BRFSS,Mental Health,Lifetime diagnosis of depression,Percentage of older adults with a lifetime dia...,%,...,POINT (-111.58713063499971 39.360700171000474),C05,TMC03,Q27,49,AGE,65PLUS,SEX,FEMALE,2015_2015_49_Q27_65PLUS_FEMALE
19,BRFSS~2015~2015~33~Q43~TOC11~AGE~SEX,2015,2015,NH,New Hampshire,BRFSS,Overall Health,Arthritis among older adults,Percentage of older adults ever told they have...,%,...,POINT (-71.50036091999965 43.65595011300047),C01,TOC11,Q43,33,AGE,65PLUS,SEX,FEMALE,2015_2015_33_Q43_65PLUS_FEMALE
31,BRFSS~2015~2015~29~Q27~TMC03~AGE~SEX,2015,2015,MO,Missouri,BRFSS,Mental Health,Lifetime diagnosis of depression,Percentage of older adults with a lifetime dia...,%,...,POINT (-92.56630005299968 38.635790776000476),C05,TMC03,Q27,29,AGE,65PLUS,SEX,FEMALE,2015_2015_29_Q27_65PLUS_FEMALE
84,BRFSS~2015~2015~37~Q03~TMC01~AGE~SEX,2015,2015,NC,North Carolina,BRFSS,Mental Health,Frequent mental distress,Percentage of older adults who are experiencin...,%,...,POINT (-79.15925046299964 35.466220975000454),C05,TMC01,Q03,37,AGE,65PLUS,SEX,FEMALE,2015_2015_37_Q03_65PLUS_FEMALE
129,BRFSS~2016~2016~06~Q27~TMC03~AGE~SEX,2016,2016,CA,California,BRFSS,Mental Health,Lifetime diagnosis of depression,Percentage of older adults with a lifetime dia...,%,...,POINT (-120.99999953799971 37.63864012300047),C05,TMC03,Q27,6,AGE,65PLUS,SEX,FEMALE,2016_2016_6_Q27_65PLUS_FEMALE


In [41]:
# Remove unnecessary records

# Keep records where YearStart and YearEnd are the same
clean = clean[
    clean["YearStart"] == clean["YearEnd"]
].copy()

# Keep percentage measurements
clean = clean[
    clean["DataValueTypeID"] == "PRCTG"
].copy()

# Remove missing Data_Value records
clean = clean.dropna(subset=["Data_Value"]).copy()

print("Cleaned dataset shape:", clean.shape)
print("Missing Data_Value:", clean["Data_Value"].isna().sum())

display(clean.head())

Cleaned dataset shape: (8544, 32)
Missing Data_Value: 0


,RowId,YearStart,YearEnd,LocationAbbr,LocationDesc,Datasource,Class,Topic,Question,Data_Value_Unit,...,Geolocation,ClassID,TopicID,QuestionID,LocationID,StratificationCategoryID1,StratificationID1,StratificationCategoryID2,StratificationID2,RecordID
11,BRFSS~2015~2015~49~Q27~TMC03~AGE~SEX,2015,2015,UT,Utah,BRFSS,Mental Health,Lifetime diagnosis of depression,Percentage of older adults with a lifetime dia...,%,...,POINT (-111.58713063499971 39.360700171000474),C05,TMC03,Q27,49,AGE,65PLUS,SEX,FEMALE,2015_2015_49_Q27_65PLUS_FEMALE
19,BRFSS~2015~2015~33~Q43~TOC11~AGE~SEX,2015,2015,NH,New Hampshire,BRFSS,Overall Health,Arthritis among older adults,Percentage of older adults ever told they have...,%,...,POINT (-71.50036091999965 43.65595011300047),C01,TOC11,Q43,33,AGE,65PLUS,SEX,FEMALE,2015_2015_33_Q43_65PLUS_FEMALE
31,BRFSS~2015~2015~29~Q27~TMC03~AGE~SEX,2015,2015,MO,Missouri,BRFSS,Mental Health,Lifetime diagnosis of depression,Percentage of older adults with a lifetime dia...,%,...,POINT (-92.56630005299968 38.635790776000476),C05,TMC03,Q27,29,AGE,65PLUS,SEX,FEMALE,2015_2015_29_Q27_65PLUS_FEMALE
84,BRFSS~2015~2015~37~Q03~TMC01~AGE~SEX,2015,2015,NC,North Carolina,BRFSS,Mental Health,Frequent mental distress,Percentage of older adults who are experiencin...,%,...,POINT (-79.15925046299964 35.466220975000454),C05,TMC01,Q03,37,AGE,65PLUS,SEX,FEMALE,2015_2015_37_Q03_65PLUS_FEMALE
129,BRFSS~2016~2016~06~Q27~TMC03~AGE~SEX,2016,2016,CA,California,BRFSS,Mental Health,Lifetime diagnosis of depression,Percentage of older adults with a lifetime dia...,%,...,POINT (-120.99999953799971 37.63864012300047),C05,TMC03,Q27,6,AGE,65PLUS,SEX,FEMALE,2016_2016_6_Q27_65PLUS_FEMALE


In [42]:
# Calculate percentile rank for each health indicator
clean["PercentileRank"] = (
    clean.groupby("QuestionID")["Data_Value"]
    .rank(pct=True)
)

# Categorize values into Low, Medium, and High
clean["Category"] = pd.cut(
    clean["PercentileRank"],
    bins=[0, 1/3, 2/3, 1],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

# Create the item label
clean["Item"] = (
    clean["QuestionID"].astype(str)
    + ": "
    + clean["Category"].astype(str)
)

display(
    clean[[
        "QuestionID",
        "Question",
        "Data_Value",
        "PercentileRank",
        "Category",
        "Item"
    ]].head()
)

,QuestionID,Question,Data_Value,PercentileRank,Category,Item
11,Q27,Percentage of older adults with a lifetime dia...,23.4,0.966749,High,Q27: High
19,Q43,Percentage of older adults ever told they have...,58.7,0.736842,High,Q43: High
31,Q27,Percentage of older adults with a lifetime dia...,20.1,0.759852,High,Q27: High
84,Q03,Percentage of older adults who are experiencin...,9.4,0.647783,Medium,Q03: Medium
129,Q27,Percentage of older adults with a lifetime dia...,16.8,0.406404,Medium,Q27: Medium


In [ ]:
#Create TransactionID - Each transaction represents one state in one year

clean["TransactionID"] = (
    clean["LocationAbbr"].astype(str)
    + "_"
    + clean["YearStart"].astype(str)
)

print("Number of transactions:", clean["TransactionID"].nunique())
print("Number of health indicators:", clean["QuestionID"].nunique())

# Check duplicate indicators within each transaction
print(
    "Duplicate TransactionID and QuestionID:",
    clean.duplicated(
        subset=["TransactionID", "QuestionID"]
    ).sum()
)

Number of transactions: 406
Number of health indicators: 36
Duplicate TransactionID and QuestionID: 0


In [44]:
# Item table - shows which health indicator items belong to each state-year transaction

item_table = clean[
    [
        "TransactionID",
        "LocationAbbr",
        "YearStart",
        "QuestionID",
        "Question",
        "Data_Value",
        "Category",
        "Item"
    ]
].copy()

print("Item table shape:", item_table.shape)

display(item_table.head())

Item table shape: (8544, 8)


,TransactionID,LocationAbbr,YearStart,QuestionID,Question,Data_Value,Category,Item
11,UT_2015,UT,2015,Q27,Percentage of older adults with a lifetime dia...,23.4,High,Q27: High
19,NH_2015,NH,2015,Q43,Percentage of older adults ever told they have...,58.7,High,Q43: High
31,MO_2015,MO,2015,Q27,Percentage of older adults with a lifetime dia...,20.1,High,Q27: High
84,NC_2015,NC,2015,Q03,Percentage of older adults who are experiencin...,9.4,Medium,Q03: Medium
129,CA_2016,CA,2016,Q27,Percentage of older adults with a lifetime dia...,16.8,Medium,Q27: Medium


In [45]:
# wide categorical table - each row represents a transaction, and each column represents a health indicator

wide_table = item_table.pivot(
    index="TransactionID",
    columns="QuestionID",
    values="Category"
)

wide_table = wide_table.reset_index()

print("Wide table shape:", wide_table.shape)

display(wide_table.head())



Wide table shape: (406, 37)


QuestionID,TransactionID,Q01,Q02,Q03,Q04,Q05,Q07,Q09,Q11,Q12,...,Q37,Q38,Q39,Q40,Q41,Q42,Q43,Q44,Q45,Q46
0,AK_2015,Medium,High,High,Low,NaN,NaN,Low,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,High,NaN,Medium,NaN
1,AK_2016,NaN,NaN,Low,NaN,High,Medium,Low,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,High,NaN,NaN,NaN,Low
2,AK_2017,Low,Medium,Low,Low,NaN,NaN,Medium,NaN,NaN,...,High,High,Medium,NaN,NaN,NaN,High,Low,High,High
3,AK_2018,NaN,NaN,Medium,NaN,Medium,Medium,Low,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medium
4,AK_2019,Medium,High,Low,Low,NaN,NaN,Low,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Low,Low,Medium,Low


In [46]:
# True/False transaction matrix

# Save transaction IDs
transaction_ids = wide_table["TransactionID"]

# Select indicator columns
question_columns = wide_table.drop(
    columns=["TransactionID"]
)

# Convert categories into binary columns
basket = pd.get_dummies(
    question_columns,
    prefix_sep=": "
).astype(bool)

# Add TransactionID as the index
basket.index = transaction_ids
basket.index.name = "TransactionID"

print("Transaction matrix shape:", basket.shape)

display(basket.head())

Transaction matrix shape: (406, 108)


,Q01: Low,Q01: Medium,Q01: High,Q02: Low,Q02: Medium,Q02: High,Q03: Low,Q03: Medium,Q03: High,Q04: Low,...,Q43: High,Q44: Low,Q44: Medium,Q44: High,Q45: Low,Q45: Medium,Q45: High,Q46: Low,Q46: Medium,Q46: High
TransactionID,,,,,,,,,,,,,,,,,,,,,
AK_2015,False,True,False,False,False,True,False,False,True,True,...,True,False,False,False,False,True,False,False,False,False
AK_2016,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
AK_2017,True,False,False,False,True,False,True,False,False,True,...,True,True,False,False,False,False,True,False,False,True
AK_2018,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,True,False
AK_2019,False,True,False,False,False,True,True,False,False,True,...,False,True,False,False,False,True,False,True,False,False


In [47]:
# Convert the basket into a long-format item list

basket_reset = basket.reset_index()

long_items = basket_reset.melt(
    id_vars="TransactionID",
    var_name="Item",
    value_name="Present"
)

# Keep only items that are True
long_items = long_items[
    long_items["Present"] == True
].drop(columns=["Present"])

print("Long item list shape:", long_items.shape)

display(long_items.head())

Long item list shape: (8544, 2)


,TransactionID,Item
2,AK_2017,Q01: Low
8,AL_2015,Q01: Low
10,AL_2017,Q01: Low
12,AL_2019,Q01: Low
14,AL_2021,Q01: Low


In [51]:
# transaction matrix using crosstab

rebuilt_basket = pd.crosstab(
    long_items["TransactionID"],
    long_items["Item"]
)

# Match the original matrix dimensions and order
rebuilt_basket = rebuilt_basket.reindex(
    index=basket.index,
    columns=basket.columns,
    fill_value=0
).astype(bool)

# Display the rebuilt transaction matrix
display(rebuilt_basket)

,Q01: Low,Q01: Medium,Q01: High,Q02: Low,Q02: Medium,Q02: High,Q03: Low,Q03: Medium,Q03: High,Q04: Low,...,Q43: High,Q44: Low,Q44: Medium,Q44: High,Q45: Low,Q45: Medium,Q45: High,Q46: Low,Q46: Medium,Q46: High
TransactionID,,,,,,,,,,,,,,,,,,,,,
AK_2015,False,True,False,False,False,True,False,False,True,True,...,True,False,False,False,False,True,False,False,False,False
AK_2016,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,True,False,False
AK_2017,True,False,False,False,True,False,True,False,False,True,...,True,True,False,False,False,False,True,False,False,True
AK_2018,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,True,False
AK_2019,False,True,False,False,False,True,True,False,False,True,...,False,True,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WY_2018,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
WY_2019,False,False,True,False,True,False,True,False,False,True,...,False,False,True,False,True,False,False,False,True,False
WY_2020,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,True,False,False,True,False,False
